# B1 · Preprocessing a mixed dataset, one column type at a time

**Type**: Implement with scikit-learn · **Level**: ●○○ guided · **Time**: about 35 minutes · **core path ★**

## Objective

Turn a dataset with numeric, nominal and ordinal columns into a numeric matrix, using the right scaler and the two
encoders for each group, keeping the column names, and train a logistic regression.

## Dataset

`ds.loans(with_missing=False)`: four numeric columns (`age`, `income`, `job_tenure`, `amount`), two nominal ones
(`city`, 12 levels, and `contract_type`, 4), one ordinal (`education`, with order `ds.EDUCATION_ORDER`) and the target
`defaulted`. In module 3 you will see how to do it in one line with `ColumnTransformer`: here you do it by hand to understand what happens.

## Educational purpose

Practice `fit` on the training set and `transform` on the test set for three different transformers, and keep track of which column
of the final matrix corresponds to which variable: without the names, the coefficients of a model are unreadable.

## Task

1. Split into training and test (75/25, `stratify=y`, `random_state=0`).
2. Standardize the numeric columns: `Z_num_tr`, `Z_num_te` (scaler fitted only on the training set).
3. Encode the nominal ones with `OneHotEncoder(sparse=False, handle_unknown="ignore")`: `Z_nom_tr`, `Z_nom_te`.
4. Encode the ordinal one with `OrdinalEncoder(categories=[ds.EDUCATION_ORDER])`: `Z_ord_tr`, `Z_ord_te`.
5. Join the three blocks with `np.hstack` in `Z_tr` and `Z_te`, and build `column_names` (the names of the 21 columns, in order: numeric, one-hot with `get_feature_names_out`, ordinal).
6. Train `LogisticRegression(max_iter=1000)` on `Z_tr`, compute `test_auc` and save in `top3` the names of the three columns with the largest coefficient **in absolute value**.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = ds.loans(with_missing=False)
NUM = ["age", "income", "job_tenure", "amount"]
NOMINAL = ["city", "contract_type"]
ORDINAL = ["education"]
X = df[NUM + NOMINAL + ORDINAL]
y = df["defaulted"]

## Your work

In [ ]:
X_tr, X_te, y_tr, y_te = ...

In [ ]:
scaler = StandardScaler()
Z_num_tr = ...
Z_num_te = ...

In [ ]:
ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")
Z_nom_tr = ...
Z_nom_te = ...

oe = OrdinalEncoder(categories=[ds.EDUCATION_ORDER])
Z_ord_tr = ...
Z_ord_te = ...

In [ ]:
Z_tr = ...
Z_te = ...
column_names = ...
Z_tr.shape

In [ ]:
model = LogisticRegression(max_iter=1000)
test_auc = ...
top3 = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For the one-hot names: `list(ohe.get_feature_names_out(NOMINAL))`. For top3: `pd.Series(model.coef_[0], index=column_names).abs().nlargest(3).index.tolist()`.'):
    check(len(X_te) == 375 and abs(y_te.mean() - y.mean()) < 0.01, "stratified 75/25 split", "use test_size=0.25 and stratify=y")
    check(Z_tr.shape[1] == 4 + 12 + 4 + 1 and Z_te.shape[1] == Z_tr.shape[1], "21 columns in both training and test",
          "4 numeric + 12 cities + 4 contracts + 1 ordinal = 21")
    check(np.abs(Z_tr[:, :4].mean(axis=0)).max() < 1e-9 and np.abs(Z_te[:, :4].mean(axis=0)).max() > 1e-3,
          "the numeric columns are centered on the training set, not on the test set", "fit the scaler only on X_tr, then transform X_te")
    check(sorted(np.unique(Z_tr[:, -1])) == [0, 1, 2, 3, 4], "the ordinal uses the codes 0-4 in the right order",
          "use OrdinalEncoder(categories=[ds.EDUCATION_ORDER])")
    check(set(np.unique(Z_tr[:, 4:20])) <= {0.0, 1.0}, "the one-hot columns contain only 0 and 1", "columns 4-19 must be one-hot")
    check(len(column_names) == 21 and column_names[:4] == NUM and column_names[-1] == "education" and "contract_type_unemployed" in column_names,
          "column_names is correct", "numeric, then one-hot (get_feature_names_out), then 'education'")
    check(test_auc > 0.65, "the model has a reasonable test AUC", "test_auc should exceed 0.65")
    check(len(top3) == 3 and set(top3) <= set(column_names), "top3 is a list of three column names", "top3 = names of the three columns with the largest |coef|")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_manual_preprocessing_mixed_dataset.ipynb`. Open it after you pass the check, or after you have really tried.